# Retrieval Evaluation

Measure **Hit Rate@k** and **MRR** for the embedding + FAISS retriever against `src/json/test_dataset.json`.

Metrics and reporting come from `src/json/evaluate_rag.py` (includes `additional_chunk_ids` and breakdowns by type, difficulty, and chapter).

## 1. Setup

In [1]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "src" / "vector_store" / "faiss.index").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected src/vector_store/faiss.index)"
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VECTOR_STORE_DIR = PROJECT_ROOT / "src" / "vector_store"
TEST_DATASET_PATH = PROJECT_ROOT / "src" / "json" / "test_dataset.json"

In [2]:
from src.embeddings.embedder import encode_texts
from src.json.evaluate_rag import build_faiss_retriever, evaluate, load_dataset
from src.vector_store.vector_store import FaissVectorStore

vector_store = FaissVectorStore.load(VECTOR_STORE_DIR)
retriever = build_faiss_retriever(vector_store, encode_texts)

print(f"Vectors in index: {vector_store.total_vectors()}")
print(f"Metadata entries: {len(vector_store.metadata_store)}")

Vectors in index: 780
Metadata entries: 780


## 2. Run evaluation (Hit Rate@k and MRR)

In [3]:
TOP_K = 5

test_data = load_dataset(TEST_DATASET_PATH)
results = evaluate(retriever, top_k=TOP_K, dataset=test_data, verbose=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== RAG Retrieval Evaluation (top_k=5) ===
Overall  -> MRR: 0.7710   Hit Rate@5: 0.9500   (n=100)

By difficulty:
  easy     -> MRR: 0.6667   Hit Rate@5: 1.0000   (n=2)
  hard     -> MRR: 0.8022   Hit Rate@5: 0.9839   (n=62)
  medium   -> MRR: 0.7231   Hit Rate@5: 0.8889   (n=36)

By question type:
  definition-distinction -> MRR: 0.8571   Hit Rate@5: 0.9286   (n=14)
  factual                -> MRR: 0.6681   Hit Rate@5: 0.9130   (n=23)
  multi-hop              -> MRR: 0.8556   Hit Rate@5: 1.0000   (n=9)
  negation               -> MRR: 0.7593   Hit Rate@5: 0.8889   (n=9)
  numerical              -> MRR: 0.7822   Hit Rate@5: 0.9778   (n=45)

By chapter:
  CH1      -> MRR: 0.8889   Hit Rate@5: 1.0000   (n=12)
  CH10     -> MRR: 0.6500   Hit Rate@5: 0.8750   (n=8)
  CH2      -> MRR: 0.8000   Hit Rate@5: 0.9000   (n=10)
  CH3      -> MRR: 0.7333   Hit Rate@5: 0.9000   (n=10)
  CH4      -> MRR: 0.8750   Hit Rate@5: 1.0000   (n=12)
  CH5      -> MRR: 0.6597   Hit Rate@5: 0.9167   (n=12)
  C

In [4]:
# {
#     "top_k": results["top_k"],
#     "num_queries": overall["n"],
#     "hit_rate_at_k": overall["hit_rate"],
#     "mrr": overall["mrr"],
# }overall = results["overall"]


SyntaxError: invalid syntax (1923591597.py, line 6)

## 3. Optional: compare multiple k values

In [5]:
k_sweep = {}
for k in (1, 3, 5, 10):
    run = evaluate(retriever, top_k=k, dataset=test_data, verbose=False)
    k_sweep[k] = run["overall"]

k_sweep

{1: {'n': 100, 'mrr': 0.64, 'hit_rate': 0.64},
 3: {'n': 100, 'mrr': 0.76, 'hit_rate': 0.9},
 5: {'n': 100, 'mrr': 0.771, 'hit_rate': 0.95},
 10: {'n': 100, 'mrr': 0.7738, 'hit_rate': 0.97}}

## 4. Misses (queries with no relevant chunk in top-k)

In [6]:
misses = [
    row
    for row in results["per_question"]
    if row["hit"] == 0.0
]

for row in misses:
    q = next(item for item in test_data if item["id"] == row["id"])
    print(f"#{row['id']} [{row['type']}, {row['difficulty']}]")
    print(f"  Q: {q['question']}")
    print(f"  Relevant: {row['relevant']}")
    print(f"  Retrieved@{TOP_K}: {row['retrieved']}")
    print()

#22 [numerical, hard]
  Q: Compare the client-level and member-level open position limits for commodity Index Options versus Index Futures — by what factor are the Index Options limits higher?
  Relevant: ['170', '151']
  Retrieved@5: ['150', '467', '586', '337', '147']

#27 [definition-distinction, medium]
  Q: What term describes the rupee benefit a processor gets from holding physical inventory of a commodity rather than holding a futures contract on it?
  Relevant: ['196']
  Retrieved@5: ['197', '62', '179', '125', '375']

#56 [negation, medium]
  Q: Between a long straddle and a short straddle, which one has unlimited maximum risk rather than risk limited to the premium?
  Relevant: ['418']
  Retrieved@5: ['416', '417', '432', '421', '252']

#68 [factual, medium]
  Q: What are the three settlement options currently available in the commodity futures market for an open position at expiry?
  Relevant: ['539']
  Retrieved@5: ['305', '644', '293', '204', '337']

#95 [factual, medium]


# 5. Report

## 5.1 Baseline Configuration
Dataset: 100 questions
Embedding model: <your model>
Vector store: FAISS IndexFlatL2
Evaluation metrics: Hit Rate@K, MRR
K values: 1, 3, 5, 10


## 5.2 Overall Results
| Top-K |    MRR | Hit Rate |
| ----: | -----: | -------: |
|     1 | 0.6400 |      64% |
|     3 | 0.7600 |      90% |
|     5 | 0.7710 |      95% |
|    10 | 0.7738 |      97% |


* Retrieval improves substantially from K=1 to K=3. Improvements after K=3 are smaller, with Hit Rate continuing to increase while MRR changes only slightly.


## 5.3 Breakdown
| Difficulty |    MRR |  Hit@5 |  N |
| ---------- | -----: | -----: | -: |
| Easy       | 0.6667 |   100% |  2 |
| Medium     | 0.7231 | 88.89% | 36 |
| Hard       | 0.8022 | 98.39% | 62 |


## 5.4 Failure Analysis
|  Q | Failure type                  | Interpretation                                    |
| -: | ----------------------------- | ------------------------------------------------- |
| 22 | Partial retrieval             | Part of the required evidence was retrieved       |
| 27 | Ground-truth boundary problem | Adjacent chunk contains the answer                |
| 56 | Near-topic retrieval          | Related material retrieved, exact evidence missed |
| 68 | Actual miss                   | Answer-bearing evidence not retrieved             |
| 95 | Near-topic retrieval          | Relevant topic retrieved, exact evidence missed   |


## 5.5 Baseline Limitations
Limitations identified:

1. Exact chunk-ID evaluation can penalize retrieval of adjacent
   answer-bearing chunks.

2. Hit Rate treats partial evidence and complete evidence similarly.

3. The current evaluation does not distinguish topic relevance
   from answer-bearing retrieval.

4. The dataset ground truth requires further evidence-level auditing.

5. Retrieval quality alone does not measure final answer quality.